# Value distributions in the full 2023 data

Second exploration notebook. `01_explore_samples.ipynb` looked at 5,000-row samples to
understand the columns; this one runs against the **full** 2023 files in `data/raw/`
(Part B 9,660,647 rows / 2.9 GB, Part D 26,794,878 rows / 3.6 GB).

Two questions to answer:

1. **What should "outlier" mean numerically?** How spread out are spend and utilization,
   overall and within a peer group (same specialty, same procedure/drug)? Where do the
   distributions have long tails, and how small do peer groups get before a percentile
   stops meaning anything?
2. **What are the sort and partition keys?** Per
   [ADR 0001](../docs/adr/0001-duckdb-parquet-storage.md), the Parquet layout is sorted
   to get zone-map pruning. That decision needs measured cardinality per specialty and
   per code, compared against the Parquet row-group size.

Everything here queries the CSVs **in place** — no load step, no schema declared up
front. That is the main practical reason DuckDB was chosen, and it is also why this
notebook is slow: every query re-parses CSV. Making that fast is lever 1 of the
optimization writeup, and this notebook is its "before" measurement.

## Setup

In [1]:
import time
from pathlib import Path

import duckdb

RAW = Path("../data/raw")
PART_B_CSV = RAW / "part_b_2023_full.csv"
PART_D_CSV = RAW / "part_d_2023_full.csv"

con = duckdb.connect()  # in-memory


def q(sql, label=""):
    """Run SQL, print wall-clock time, return a DataFrame.

    Timings are recorded deliberately: they are the CSV baseline that the
    Parquet conversion gets compared against.
    """
    t0 = time.perf_counter()
    df = con.sql(sql).df()
    print(f"[{time.perf_counter() - t0:6.1f}s] {label}")
    return df


print("duckdb", duckdb.__version__)
con.sql(
    "SELECT current_setting('memory_limit') AS memory_limit, "
    "current_setting('threads') AS threads"
).df()

duckdb 1.5.5


,memory_limit,threads
0,19.1 GiB,15


We register the CSVs as **views**, not tables. A view stores no data — each query
against it re-reads the file. That keeps memory flat and means nothing is committed to
a schema yet.

`sample_size` controls how many rows DuckDB reads to infer column types. The default
(20,480) is a small fraction of 9.7M rows, so a column that looks numeric early and
turns messy later can be mistyped. We raise it, and check the result in the next
section rather than trusting it.

In [2]:
con.execute(f"""
    CREATE OR REPLACE VIEW part_b AS
    SELECT * FROM read_csv_auto('{PART_B_CSV}', sample_size=200000)
""")
con.execute(f"""
    CREATE OR REPLACE VIEW part_d AS
    SELECT * FROM read_csv_auto('{PART_D_CSV}', sample_size=200000)
""")
print("views created (no data read yet — views are lazy)")

views created (no data read yet — views are lazy)


## 1. What types did DuckDB infer?

This matters more than it looks. The Parquet build will write explicit types, and this
is the draft of that type list. Two things to look for:

- **Identifier columns inferred as integers.** `Rndrng_Prvdr_Zip5` and the FIPS codes
  have leading zeros. If they come back as `BIGINT`, the leading zeros are already gone
  and any join on them later will silently miss rows. These want `VARCHAR`.
- **Measure columns inferred as `VARCHAR`.** That usually means suppression blanks or a
  stray non-numeric value somewhere in the file — a data-quality finding, not a typing
  preference.

In [3]:
q("DESCRIBE part_b", "describe part_b")

[   0.8s] describe part_b


,column_name,column_type,null,key,default,extra
0,Rndrng_NPI,BIGINT,YES,None,None,None
1,Rndrng_Prvdr_Last_Org_Name,VARCHAR,YES,None,None,None
2,Rndrng_Prvdr_First_Name,VARCHAR,YES,None,None,None
3,Rndrng_Prvdr_MI,VARCHAR,YES,None,None,None
4,Rndrng_Prvdr_Crdntls,VARCHAR,YES,None,None,None
5,Rndrng_Prvdr_Ent_Cd,VARCHAR,YES,None,None,None
6,Rndrng_Prvdr_St1,VARCHAR,YES,None,None,None
7,Rndrng_Prvdr_St2,VARCHAR,YES,None,None,None
8,Rndrng_Prvdr_City,VARCHAR,YES,None,None,None
9,Rndrng_Prvdr_State_Abrvtn,VARCHAR,YES,None,None,None


In [4]:
q("DESCRIBE part_d", "describe part_d")

[   0.6s] describe part_d


,column_name,column_type,null,key,default,extra
0,Prscrbr_NPI,BIGINT,YES,None,None,None
1,Prscrbr_Last_Org_Name,VARCHAR,YES,None,None,None
2,Prscrbr_First_Name,VARCHAR,YES,None,None,None
3,Prscrbr_City,VARCHAR,YES,None,None,None
4,Prscrbr_State_Abrvtn,VARCHAR,YES,None,None,None
5,Prscrbr_State_FIPS,VARCHAR,YES,None,None,None
6,Prscrbr_Type,VARCHAR,YES,None,None,None
7,Prscrbr_Type_Src,VARCHAR,YES,None,None,None
8,Brnd_Name,VARCHAR,YES,None,None,None
9,Gnrc_Name,VARCHAR,YES,None,None,None


## 2. Scale

First full scan of each file. The row counts should match what `data/Data.md` records
from the download; if they don't, something went wrong in the pull and everything
downstream is suspect.

The timings here are the CSV baseline.

In [5]:
q("SELECT COUNT(*) AS part_b_rows FROM part_b", "part_b row count (full CSV scan)")

[   1.4s] part_b row count (full CSV scan)


,part_b_rows
0,9660647


In [6]:
q("SELECT COUNT(*) AS part_d_rows FROM part_d", "part_d row count (full CSV scan)")

[   1.2s] part_d row count (full CSV scan)


,part_d_rows
0,26794878


## 3. Cardinality

Inputs to the sort/partition key decision, and a first sense of how many distinct peer
groups the outlier scoring has to handle.

Each of these is one scan computing every count at once, rather than one scan per
count — the queries are written for the storage format we have, which is a habit worth
keeping.

In [7]:
q("""
    SELECT
        COUNT(DISTINCT Rndrng_NPI)                  AS providers,
        COUNT(DISTINCT Rndrng_Prvdr_Type)           AS specialties,
        COUNT(DISTINCT HCPCS_Cd)                    AS hcpcs_codes,
        COUNT(DISTINCT Rndrng_Prvdr_State_Abrvtn)   AS states,
        COUNT(DISTINCT Rndrng_Prvdr_RUCA)           AS ruca_codes,
        COUNT(DISTINCT Place_Of_Srvc)               AS places_of_service,
        COUNT(DISTINCT Rndrng_Prvdr_Ent_Cd)         AS entity_types
    FROM part_b
""", "part_b cardinality")

[   1.3s] part_b cardinality


,providers,specialties,hcpcs_codes,states,ruca_codes,places_of_service,entity_types
0,1175281,104,6405,62,22,2,2


In [8]:
q("""
    SELECT
        COUNT(DISTINCT Prscrbr_NPI)                 AS prescribers,
        COUNT(DISTINCT Prscrbr_Type)                AS specialties,
        COUNT(DISTINCT Gnrc_Name)                   AS generic_drugs,
        COUNT(DISTINCT Brnd_Name)                   AS brand_names,
        COUNT(DISTINCT Prscrbr_State_Abrvtn)        AS states,
        COUNT(DISTINCT Prscrbr_Type_Src)            AS specialty_sources
    FROM part_d
""", "part_d cardinality")

[   1.4s] part_d cardinality


,prescribers,specialties,generic_drugs,brand_names,states,specialty_sources
0,1104162,175,1779,3027,62,3


### 3b. How big is a peer group?

A "peer group" is the comparison set an outlier is measured against — same specialty,
same procedure or drug. Group size drives two separate decisions:

- **Statistically:** a percentile computed over 8 rows is noise. Somewhere there is a
  minimum group size below which we should refuse to score rather than publish a
  meaningless rank. These numbers say how much of the data that rule would exclude.
- **Physically:** DuckDB writes Parquet row groups of 122,880 rows by default, and
  zone-map pruning works at row-group granularity. What decides the pruning win is not
  group size on its own but **whether a filtered peer group's rows are contiguous on
  disk**. Unsorted, one group's rows are scattered across every row group and nothing
  can be skipped. Sorted by `(specialty, code)`, the rows are adjacent — and the
  *smaller* the group relative to a row group, the larger the fraction of the file a
  single-group query can skip. So small groups are what make the sort pay off, not what
  undermines it.

  (The first pass of this notebook had that backwards, and predicted small groups would
  weaken the pruning story. The measurement below is what corrected it.)

In [9]:
ROW_GROUP = 122_880  # DuckDB default Parquet row group size

q(f"""
    WITH g AS (
        SELECT Rndrng_Prvdr_Type, HCPCS_Cd, COUNT(*) AS n
        FROM part_b
        GROUP BY 1, 2
    )
    SELECT
        COUNT(*)                                        AS n_groups,
        SUM(n)                                          AS n_rows,
        ROUND(AVG(n), 1)                                AS mean_rows,
        quantile_cont(n, 0.50)                          AS p50,
        quantile_cont(n, 0.90)                          AS p90,
        quantile_cont(n, 0.99)                          AS p99,
        MAX(n)                                          AS max_rows,
        COUNT(*) FILTER (n < 30)                        AS groups_under_30,
        SUM(n)   FILTER (n < 30)                        AS rows_in_groups_under_30,
        COUNT(*) FILTER (n >= {ROW_GROUP})              AS groups_over_one_row_group
    FROM g
""", "part_b peer-group sizes (specialty x hcpcs)")

[   1.2s] part_b peer-group sizes (specialty x hcpcs)


,n_groups,n_rows,mean_rows,p50,p90,p99,max_rows,groups_under_30,rows_in_groups_under_30,groups_over_one_row_group
0,48165,9660647.0,200.6,4.0,156.0,4323.8,96640,37536,188772.0,0


In [10]:
q(f"""
    WITH g AS (
        SELECT Prscrbr_Type, Gnrc_Name, COUNT(*) AS n
        FROM part_d
        GROUP BY 1, 2
    )
    SELECT
        COUNT(*)                                        AS n_groups,
        SUM(n)                                          AS n_rows,
        ROUND(AVG(n), 1)                                AS mean_rows,
        quantile_cont(n, 0.50)                          AS p50,
        quantile_cont(n, 0.90)                          AS p90,
        quantile_cont(n, 0.99)                          AS p99,
        MAX(n)                                          AS max_rows,
        COUNT(*) FILTER (n < 30)                        AS groups_under_30,
        SUM(n)   FILTER (n < 30)                        AS rows_in_groups_under_30,
        COUNT(*) FILTER (n >= {ROW_GROUP})              AS groups_over_one_row_group
    FROM g
""", "part_d peer-group sizes (specialty x generic drug)")

[   1.3s] part_d peer-group sizes (specialty x generic drug)


,n_groups,n_rows,mean_rows,p50,p90,p99,max_rows,groups_under_30,rows_in_groups_under_30,groups_over_one_row_group
0,48802,26794878.0,549.1,5.0,324.0,13633.7,134126,35717,190608.0,1


**What the two tables above settle.**

Both datasets have the same shape: ~48,000 peer groups, a median group of **4 rows
(Part B) / 5 rows (Part D)**, and a p99 three orders of magnitude higher (4,324 /
13,634). Almost all groups are tiny; almost all *rows* live in a handful of enormous
ones.

That asymmetry is unusually convenient, and it resolves both decisions at once:

1. **A minimum-group-size rule is nearly free.** Requiring n ≥ 30 to score a provider
   drops 37,536 of 48,165 Part B groups (78%) — but only 188,772 of 9,660,647 rows
   (**1.95%**). Part D: 73% of groups, **0.71%** of rows. We can refuse to rank anyone
   against a handful of peers and still cover ~98–99% of the data. That is a rule worth
   adopting precisely because it costs so little.

2. **The sort key is confirmed, with a bound.** Part B's largest peer group is 96,640
   rows — *smaller than a single 122,880-row Parquet row group*. So once sorted by
   `(specialty, code)`, **every Part B peer group fits inside at most two row groups**,
   out of ~79 for the file. A single-peer-group query should be able to skip ~97% of
   row groups. Part D has exactly one group that exceeds a row group (134,126 rows,
   Family Practice / Metformin Hcl) against ~218 row groups total.

   This is the concrete prediction lever 2 of the optimization writeup has to test:
   *row groups scanned should fall from all of them to roughly one.* If it doesn't, the
   sort isn't being written the way we think it is.

## 4. Value distributions — what does "outlier" mean numerically?

### A note on what the Part B measure columns actually are

`Avg_Sbmtd_Chrg`, `Avg_Mdcr_Alowd_Amt`, `Avg_Mdcr_Pymt_Amt` and `Avg_Mdcr_Stdzd_Amt`
are **per-service averages**, not totals. There is no total-payment column: total
Medicare payment for a row is `Tot_Srvcs * Avg_Mdcr_Pymt_Amt`. That distinction decides
what an outlier is — a provider can be unremarkable on price and extreme on volume, or
the reverse, and those are different findings.

`Avg_Mdcr_Stdzd_Amt` is the payment with geographic cost adjustments removed, so it is
the right column for comparing providers **across** regions. `Avg_Mdcr_Pymt_Amt` is what
was actually paid.

Part D is the opposite: `Tot_Drug_Cst` is already a total, so cost per claim has to be
derived (`Tot_Drug_Cst / Tot_Clms`).

In [11]:
q("""
    SELECT
        measure,
        ROUND(quantile_cont(v, 0.50), 2)  AS p50,
        ROUND(quantile_cont(v, 0.90), 2)  AS p90,
        ROUND(quantile_cont(v, 0.99), 2)  AS p99,
        ROUND(quantile_cont(v, 0.999), 2) AS p999,
        ROUND(MAX(v), 2)                  AS max,
        ROUND(AVG(v), 2)                  AS mean,
        ROUND(AVG(v) / NULLIF(quantile_cont(v, 0.50), 0), 2) AS mean_over_median
    FROM (
        SELECT 'Tot_Benes'          AS measure, Tot_Benes::DOUBLE           AS v FROM part_b
        UNION ALL SELECT 'Tot_Srvcs',           Tot_Srvcs::DOUBLE           FROM part_b
        UNION ALL SELECT 'Avg_Sbmtd_Chrg',      Avg_Sbmtd_Chrg::DOUBLE      FROM part_b
        UNION ALL SELECT 'Avg_Mdcr_Alowd_Amt',  Avg_Mdcr_Alowd_Amt::DOUBLE  FROM part_b
        UNION ALL SELECT 'Avg_Mdcr_Pymt_Amt',   Avg_Mdcr_Pymt_Amt::DOUBLE   FROM part_b
        UNION ALL SELECT 'Avg_Mdcr_Stdzd_Amt',  Avg_Mdcr_Stdzd_Amt::DOUBLE  FROM part_b
        UNION ALL SELECT 'row_total_payment',   (Tot_Srvcs * Avg_Mdcr_Pymt_Amt)::DOUBLE FROM part_b
    )
    WHERE v IS NOT NULL
    GROUP BY measure
    ORDER BY measure
""", "part_b measure distributions")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[   9.1s] part_b measure distributions


,measure,p50,p90,p99,p999,max,mean,mean_over_median
0,Avg_Mdcr_Alowd_Amt,71.38,180.07,823.05,2913.33,5.279404e+04,106.11,1.49
1,Avg_Mdcr_Pymt_Amt,53.10,142.97,654.92,2343.79,4.205952e+04,82.99,1.56
2,Avg_Mdcr_Stdzd_Amt,53.63,138.74,651.97,2371.95,4.222933e+04,82.37,1.54
3,Avg_Sbmtd_Chrg,175.00,800.00,4172.93,17000.00,9.999999e+04,416.83,2.38
4,Tot_Benes,32.00,156.00,594.00,2905.35,8.704860e+05,85.37,2.67
5,Tot_Srvcs,43.00,317.00,2521.00,27000.00,5.208424e+06,273.85,6.37
6,row_total_payment,2301.76,17203.55,97364.56,592994.00,2.993196e+08,9701.32,4.21


`mean_over_median` is a quick skew read: at 1.0 the distribution is symmetric, and
the further above 1.0 it goes the more a few extreme rows drag the average. Anywhere
that ratio is large, **mean and standard deviation are the wrong tools** and a
z-score-style outlier rule would be measuring the tail against itself. That pushes
toward rank/percentile or robust (median/MAD) scoring — a methodology decision this
table should settle rather than assume.

In [12]:
q("""
    SELECT
        measure,
        ROUND(quantile_cont(v, 0.50), 2)  AS p50,
        ROUND(quantile_cont(v, 0.90), 2)  AS p90,
        ROUND(quantile_cont(v, 0.99), 2)  AS p99,
        ROUND(quantile_cont(v, 0.999), 2) AS p999,
        ROUND(MAX(v), 2)                  AS max,
        ROUND(AVG(v), 2)                  AS mean,
        ROUND(AVG(v) / NULLIF(quantile_cont(v, 0.50), 0), 2) AS mean_over_median
    FROM (
        SELECT 'Tot_Clms'         AS measure, Tot_Clms::DOUBLE        AS v FROM part_d
        UNION ALL SELECT 'Tot_30day_Fills',   Tot_30day_Fills::DOUBLE FROM part_d
        UNION ALL SELECT 'Tot_Day_Suply',     Tot_Day_Suply::DOUBLE   FROM part_d
        UNION ALL SELECT 'Tot_Drug_Cst',      Tot_Drug_Cst::DOUBLE    FROM part_d
        UNION ALL SELECT 'Tot_Benes',         Tot_Benes::DOUBLE       FROM part_d
        UNION ALL SELECT 'cost_per_claim',    (Tot_Drug_Cst / NULLIF(Tot_Clms, 0))::DOUBLE FROM part_d
    )
    WHERE v IS NOT NULL
    GROUP BY measure
    ORDER BY measure
""", "part_d measure distributions")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[  10.7s] part_d measure distributions


,measure,p50,p90,p99,p999,max,mean,mean_over_median
0,Tot_30day_Fills,41.00,212.00,884.90,2022.80,285620.30,95.95,2.34
1,Tot_Benes,20.00,61.00,170.00,363.00,258456.00,31.80,1.59
2,Tot_Clms,25.00,111.00,415.00,1004.00,285496.00,52.01,2.08
3,Tot_Day_Suply,1110.00,6150.00,26060.00,59219.00,1092131.00,2700.28,2.43
4,Tot_Drug_Cst,640.93,11471.40,138867.45,612129.09,81776016.73,7937.69,12.38
5,cost_per_claim,18.13,519.95,3197.96,18110.12,492792.38,229.83,12.67


### 4b. Spread *within* a peer group

The tables above mix every specialty and every code together, so most of that spread is
just "an MRI costs more than an office visit" — not interesting. Outlier detection
happens **within** a peer group, so the question is how much spread survives once
specialty and code are held fixed.

The ratio to watch is p99/p50 within a group. If it stays high, there is real
provider-to-provider variation to find. If it collapses toward 1, the peer group
explains almost everything and outliers will be rare and probably data errors.

In [13]:
q("""
    WITH sized AS (
        SELECT
            Rndrng_Prvdr_Type AS specialty,
            HCPCS_Cd          AS code,
            COUNT(*)          AS n_providers,
            quantile_cont(Avg_Mdcr_Stdzd_Amt::DOUBLE, 0.50) AS price_p50,
            quantile_cont(Avg_Mdcr_Stdzd_Amt::DOUBLE, 0.99) AS price_p99,
            quantile_cont(Tot_Srvcs::DOUBLE, 0.50)          AS srvcs_p50,
            quantile_cont(Tot_Srvcs::DOUBLE, 0.99)          AS srvcs_p99
        FROM part_b
        GROUP BY 1, 2
        HAVING COUNT(*) >= 1000
    )
    SELECT
        specialty, code, n_providers,
        ROUND(price_p50, 2) AS price_p50,
        ROUND(price_p99, 2) AS price_p99,
        ROUND(price_p99 / NULLIF(price_p50, 0), 2) AS price_p99_over_p50,
        ROUND(srvcs_p50, 1) AS srvcs_p50,
        ROUND(srvcs_p99, 1) AS srvcs_p99,
        ROUND(srvcs_p99 / NULLIF(srvcs_p50, 0), 2) AS srvcs_p99_over_p50
    FROM sized
    ORDER BY n_providers DESC
    LIMIT 25
""", "part_b within-peer-group spread, 25 largest groups")

[   1.5s] part_b within-peer-group spread, 25 largest groups


,specialty,code,n_providers,price_p50,price_p99,price_p99_over_p50,srvcs_p50,srvcs_p99,srvcs_p99_over_p50
0,Nurse Practitioner,99214,96640,74.21,86.85,1.17,77.0,798.6,10.37
1,Nurse Practitioner,99213,89681,51.60,61.08,1.18,58.0,677.0,11.67
2,Physical Therapist in Private Practice,97110,68990,17.70,20.87,1.18,506.0,4924.1,9.73
3,Family Practice,99214,61919,84.08,100.70,1.20,161.0,1300.5,8.08
4,Physical Therapist in Private Practice,97140,58493,16.07,17.43,1.08,278.0,2649.0,9.53
5,Family Practice,99213,58182,59.07,71.20,1.21,77.0,912.0,11.84
6,Physical Therapist in Private Practice,97112,56696,21.29,26.18,1.23,298.0,2726.1,9.15
7,Physician Assistant,99213,55160,52.60,61.15,1.16,67.0,904.8,13.50
8,Physical Therapist in Private Practice,97530,54039,25.79,29.23,1.13,343.0,3413.8,9.95
9,Physician Assistant,99214,51597,75.30,87.11,1.16,66.0,737.0,11.17


Two columns, two different stories. `price_p99_over_p50` is **price** variation for
an identical procedure — for many codes Medicare largely fixes this, so a high ratio is
worth a second look. `srvcs_p99_over_p50` is **volume** variation, which is where the
genuinely interesting utilization outliers live and where the spread should be much
wider.

In [14]:
q("""
    WITH sized AS (
        SELECT
            Prscrbr_Type AS specialty,
            Gnrc_Name    AS generic,
            COUNT(*)     AS n_prescribers,
            quantile_cont(Tot_Clms::DOUBLE, 0.50) AS clms_p50,
            quantile_cont(Tot_Clms::DOUBLE, 0.99) AS clms_p99,
            quantile_cont((Tot_Drug_Cst / NULLIF(Tot_Clms, 0))::DOUBLE, 0.50) AS cpc_p50,
            quantile_cont((Tot_Drug_Cst / NULLIF(Tot_Clms, 0))::DOUBLE, 0.99) AS cpc_p99
        FROM part_d
        GROUP BY 1, 2
        HAVING COUNT(*) >= 1000
    )
    SELECT
        specialty, generic, n_prescribers,
        ROUND(clms_p50, 1) AS clms_p50,
        ROUND(clms_p99, 1) AS clms_p99,
        ROUND(clms_p99 / NULLIF(clms_p50, 0), 2) AS clms_p99_over_p50,
        ROUND(cpc_p50, 2)  AS cost_per_claim_p50,
        ROUND(cpc_p99, 2)  AS cost_per_claim_p99
    FROM sized
    ORDER BY n_prescribers DESC
    LIMIT 25
""", "part_d within-peer-group spread, 25 largest groups")

[   1.6s] part_d within-peer-group spread, 25 largest groups


,specialty,generic,n_prescribers,clms_p50,clms_p99,clms_p99_over_p50,cost_per_claim_p50,cost_per_claim_p99
0,Family Practice,Metformin Hcl,134126,55.0,428.0,7.78,10.98,37.03
1,Family Practice,Levothyroxine Sodium,113109,68.0,762.0,11.21,18.10,131.96
2,Dentist,Amoxicillin,100575,31.0,216.0,6.97,3.42,7.17
3,Nurse Practitioner,Metformin Hcl,98811,35.0,274.0,7.83,10.07,37.71
4,Family Practice,Albuterol Sulfate,97065,35.0,261.0,7.46,37.42,120.44
5,Internal Medicine,Metformin Hcl,96305,61.0,556.0,9.11,11.12,56.55
6,Nurse Practitioner,Gabapentin,90989,42.0,482.0,11.48,17.14,51.94
7,Nurse Practitioner,Atorvastatin Calcium,89911,70.0,609.0,8.70,13.64,37.06
8,Internal Medicine,Levothyroxine Sodium,88636,72.0,936.0,13.00,18.92,136.70
9,Nurse Practitioner,Levothyroxine Sodium,88477,45.0,453.2,10.07,15.46,123.11


## 5. Nulls and suppression at full scale

`01_explore_samples.ipynb` found the suppression pattern in 5,000 rows. The question
here is what it looks like across all 26.8M — specifically how much of Part D is
unusable for patient-count-based measures, and whether Part B has nulls the sample
missed.

CMS blanks `Tot_Benes` when fewer than 11 patients are involved, to prevent
re-identification. That is not random missingness: it is **systematically the small
providers**. Any metric built on `Tot_Benes` therefore silently drops the low end of
the distribution, which is exactly the kind of thing that has to be stated in the
writeup rather than discovered by a reader.

In [15]:
q("""
    SELECT
        COUNT(*)                                            AS rows,
        COUNT(*) FILTER (Tot_Benes IS NULL)                 AS tot_benes_null,
        ROUND(100.0 * COUNT(*) FILTER (Tot_Benes IS NULL) / COUNT(*), 2) AS tot_benes_null_pct,
        COUNT(*) FILTER (GE65_Sprsn_Flag IS NOT NULL)       AS ge65_suppressed,
        COUNT(*) FILTER (GE65_Bene_Sprsn_Flag IS NOT NULL)  AS ge65_bene_suppressed,
        COUNT(*) FILTER (Tot_Clms IS NULL)                  AS tot_clms_null,
        COUNT(*) FILTER (Tot_Drug_Cst IS NULL)              AS tot_drug_cst_null
    FROM part_d
""", "part_d suppression at full scale")

[   1.3s] part_d suppression at full scale


,rows,tot_benes_null,tot_benes_null_pct,ge65_suppressed,ge65_bene_suppressed,tot_clms_null,tot_drug_cst_null
0,26794878,14757509,55.08,12070819,23700772,0,0


In [16]:
q("""
    SELECT
        COUNT(*)                                          AS rows,
        COUNT(*) - COUNT(Tot_Benes)                       AS tot_benes_null,
        COUNT(*) - COUNT(Tot_Srvcs)                       AS tot_srvcs_null,
        COUNT(*) - COUNT(Avg_Mdcr_Stdzd_Amt)              AS stdzd_amt_null,
        COUNT(*) - COUNT(Rndrng_Prvdr_Type)               AS specialty_null,
        COUNT(*) - COUNT(Rndrng_Prvdr_State_Abrvtn)       AS state_null,
        COUNT(*) - COUNT(Rndrng_Prvdr_RUCA)               AS ruca_null,
        COUNT(*) FILTER (Rndrng_Prvdr_Cntry <> 'US')      AS non_us_rows
    FROM part_b
""", "part_b nulls at full scale")

[   1.3s] part_b nulls at full scale


,rows,tot_benes_null,tot_srvcs_null,stdzd_amt_null,specialty_null,state_null,ruca_null,non_us_rows
0,9660647,0,0,0,0,0,7600,395


## 6. Peer-key variants — does geography belong in the peer key?

Everything above defines a peer group as `(specialty, code)`. That is not obviously
right. "Compared against others in your state" is the more intuitive claim, and it is
what a reader of the dashboard will assume unless told otherwise.

The argument against is arithmetic: the median peer group is already 4–5 rows. Splitting
by state multiplies group count by up to 62 and pushes most groups below the point where
a percentile means anything.

The argument for leaving geography out is that `Avg_Mdcr_Stdzd_Amt` exists **precisely**
to remove geographic cost adjustment, so cross-region price comparison is already valid
without partitioning on state. That argument covers price. It does not cover volume —
whether a provider sees an unusual number of patients is plausibly a local phenomenon.

So this is a real tradeoff, and the numbers below decide it. The column to watch is
`pct_rows_in_groups_under_30`: how much of the dataset becomes unscoreable under each
definition.

In [17]:
import pandas as pd


def group_size_profile(view, keys, label):
    """One row summarising the peer-group size distribution under a given key."""
    return con.sql(f"""
        WITH g AS (
            SELECT {", ".join(keys)}, COUNT(*) AS n
            FROM {view}
            GROUP BY ALL
        )
        SELECT
            '{label}'                                              AS peer_key,
            COUNT(*)                                               AS n_groups,
            ROUND(AVG(n), 1)                                       AS mean_rows,
            quantile_cont(n, 0.50)                                 AS p50,
            quantile_cont(n, 0.90)                                 AS p90,
            MAX(n)                                                 AS max_rows,
            ROUND(100.0 * COUNT(*) FILTER (n < 30) / COUNT(*), 1)  AS pct_groups_under_30,
            ROUND(100.0 * SUM(n)   FILTER (n < 30) / SUM(n), 2)    AS pct_rows_under_30
        FROM g
    """).df()


t0 = time.perf_counter()
part_b_keys = [
    (["Rndrng_Prvdr_Type", "HCPCS_Cd"], "specialty + code"),
    (["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"], "specialty + code + place of service"),
    (["Rndrng_Prvdr_Type", "HCPCS_Cd", "Rndrng_Prvdr_RUCA"], "specialty + code + RUCA"),
    (["Rndrng_Prvdr_Type", "HCPCS_Cd", "Rndrng_Prvdr_State_Abrvtn"], "specialty + code + state"),
]
out = pd.concat(
    [group_size_profile("part_b", k, l) for k, l in part_b_keys], ignore_index=True
)
print(f"[{time.perf_counter() - t0:6.1f}s] part_b peer-key variants")
out

[   4.8s] part_b peer-key variants


,peer_key,n_groups,mean_rows,p50,p90,max_rows,pct_groups_under_30,pct_rows_under_30
0,specialty + code,48165,200.6,4.0,156.0,96640,77.9,1.95
1,specialty + code + place of service,59087,163.5,3.0,133.0,82339,79.3,2.40
2,specialty + code + RUCA,124554,77.6,2.0,49.0,75343,86.6,4.84
3,specialty + code + state,462739,20.9,2.0,34.0,6252,88.9,19.35


In [18]:
t0 = time.perf_counter()
part_d_keys = [
    (["Prscrbr_Type", "Gnrc_Name"], "specialty + generic"),
    (["Prscrbr_Type", "Brnd_Name"], "specialty + brand"),
    (["Prscrbr_Type", "Gnrc_Name", "Prscrbr_State_Abrvtn"], "specialty + generic + state"),
]
out = pd.concat(
    [group_size_profile("part_d", k, l) for k, l in part_d_keys], ignore_index=True
)
print(f"[{time.perf_counter() - t0:6.1f}s] part_d peer-key variants")
out

[   3.9s] part_d peer-key variants


,peer_key,n_groups,mean_rows,p50,p90,max_rows,pct_groups_under_30,pct_rows_under_30
0,specialty + generic,48802,549.1,5.0,324.0,134126,73.2,0.71
1,specialty + brand,63048,425.0,4.0,236.0,100575,75.8,0.91
2,specialty + generic + state,588528,45.5,3.0,64.0,14951,84.1,8.75


## 7. Do Part B and Part D describe the same providers?

This decides a schema question: **one provider dimension, or two?** If the NPI sets
overlap heavily, there is a single `provider` table that both fact tables reference. If
they barely overlap, they are two populations that happen to share an identifier format,
and forcing them into one table would invent a relationship that isn't there.

Part B has 1,175,281 providers and Part D 1,104,162 prescribers, but those totals say
nothing about overlap on their own.

The second question is the specialty crosswalk. Part B uses 104 specialty values and
Part D 175. For providers present in both, do the strings agree? Wherever they don't,
the mismatches are the crosswalk we would have to build.

In [19]:
q("""
    WITH b AS (SELECT DISTINCT Rndrng_NPI AS npi FROM part_b),
         d AS (SELECT DISTINCT Prscrbr_NPI AS npi FROM part_d)
    SELECT
        (SELECT COUNT(*) FROM b)                                          AS part_b_providers,
        (SELECT COUNT(*) FROM d)                                          AS part_d_prescribers,
        (SELECT COUNT(*) FROM (SELECT npi FROM b INTERSECT SELECT npi FROM d)) AS in_both,
        (SELECT COUNT(*) FROM (SELECT npi FROM b EXCEPT    SELECT npi FROM d)) AS part_b_only,
        (SELECT COUNT(*) FROM (SELECT npi FROM d EXCEPT    SELECT npi FROM b)) AS part_d_only
""", "NPI overlap between Part B and Part D")

[   2.4s] NPI overlap between Part B and Part D


,part_b_providers,part_d_prescribers,in_both,part_b_only,part_d_only
0,1175281,1104162,706614,468667,397548


Before comparing specialties across datasets, check whether a provider even has one
specialty *within* a dataset. If an NPI carries several, "the provider's specialty" is
not a well-defined attribute and specialty cannot live on a provider dimension — it
belongs on the fact row.

In [20]:
q("""
    SELECT 'part_b' AS dataset, COUNT(*) AS providers,
           COUNT(*) FILTER (n_specialties > 1) AS with_multiple_specialties,
           MAX(n_specialties) AS max_specialties
    FROM (SELECT Rndrng_NPI, COUNT(DISTINCT Rndrng_Prvdr_Type) AS n_specialties
          FROM part_b GROUP BY 1)
    UNION ALL
    SELECT 'part_d', COUNT(*),
           COUNT(*) FILTER (n_specialties > 1),
           MAX(n_specialties)
    FROM (SELECT Prscrbr_NPI, COUNT(DISTINCT Prscrbr_Type) AS n_specialties
          FROM part_d GROUP BY 1)
""", "providers carrying more than one specialty within a dataset")

[   2.4s] providers carrying more than one specialty within a dataset


,dataset,providers,with_multiple_specialties,max_specialties
0,part_b,1175281,0,1
1,part_d,1104162,0,1


In [21]:
q("""
    WITH b AS (SELECT DISTINCT Rndrng_NPI AS npi, Rndrng_Prvdr_Type AS specialty_b FROM part_b),
         d AS (SELECT DISTINCT Prscrbr_NPI AS npi, Prscrbr_Type    AS specialty_d FROM part_d)
    SELECT
        COUNT(*)                                                        AS npi_specialty_pairs,
        COUNT(*) FILTER (specialty_b = specialty_d)                     AS exact_string_match,
        ROUND(100.0 * COUNT(*) FILTER (specialty_b = specialty_d) / COUNT(*), 1) AS pct_match
    FROM b JOIN d USING (npi)
""", "specialty agreement across datasets")

[   2.4s] specialty agreement across datasets


,npi_specialty_pairs,exact_string_match,pct_match
0,706614,706614,100.0


In [22]:
q("""
    WITH b AS (SELECT DISTINCT Rndrng_NPI AS npi, Rndrng_Prvdr_Type AS specialty_b FROM part_b),
         d AS (SELECT DISTINCT Prscrbr_NPI AS npi, Prscrbr_Type    AS specialty_d FROM part_d)
    SELECT specialty_b, specialty_d, COUNT(*) AS n_providers
    FROM b JOIN d USING (npi)
    WHERE specialty_b <> specialty_d
    GROUP BY 1, 2
    ORDER BY n_providers DESC
    LIMIT 30
""", "most common cross-dataset specialty mismatches (draft crosswalk)")

[   2.4s] most common cross-dataset specialty mismatches (draft crosswalk)


,specialty_b,specialty_d,n_providers


## 8. Part B's invisible suppression

Section 5 found zero nulls anywhere in Part B. That is not because Part B is unsuppressed
— it is because Part B suppresses by **removing the row**, where Part D blanks a column.

CMS's stated rule is that Part B rows with fewer than 11 beneficiaries are not published.
If that holds, `MIN(Tot_Benes)` should be exactly 11 and there should be a hard cliff at
11 with nothing below it — a distribution that has been cut, not one that tapers.

This matters because it is a *silent* bias. A provider who performs a procedure 8 times
does not appear as a null; they appear as a provider who does not perform that procedure
at all. Low-volume providers are systematically under-represented and nothing in the file
says so. The dashboard has to state this, because a user looking at a provider's profile
is seeing a censored record without being told.

In [23]:
q("""
    SELECT
        MIN(Tot_Benes)                        AS min_tot_benes,
        MIN(Tot_Srvcs)                        AS min_tot_srvcs,
        MIN(Tot_Bene_Day_Srvcs)               AS min_bene_day_srvcs,
        COUNT(*) FILTER (Tot_Benes < 11)      AS rows_below_11_benes
    FROM part_b
""", "part_b suppression floor")

[   1.2s] part_b suppression floor


,min_tot_benes,min_tot_srvcs,min_bene_day_srvcs,rows_below_11_benes
0,11,5.5,11,0


In [24]:
q("""
    SELECT Tot_Benes, COUNT(*) AS n_rows
    FROM part_b
    WHERE Tot_Benes <= 20
    GROUP BY 1
    ORDER BY 1
""", "part_b low end of the beneficiary distribution — look for the cliff at 11")

[   1.2s] part_b low end of the beneficiary distribution — look for the cliff at 11


,Tot_Benes,n_rows
0,11,477670
1,12,423446
2,13,380812
3,14,343630
4,15,314142
5,16,286713
6,17,264645
7,18,244282
8,19,226542
9,20,210145


In [25]:
q("""
    SELECT
        MIN(Tot_Clms)                      AS min_tot_clms,
        COUNT(*) FILTER (Tot_Clms < 11)    AS rows_below_11_clms,
        MIN(Tot_Benes)                     AS min_tot_benes_when_present
    FROM part_d
""", "part_d suppression floor (row-level threshold is on claims, not beneficiaries)")

[   1.2s] part_d suppression floor (row-level threshold is on claims, not beneficiaries)


,min_tot_clms,rows_below_11_clms,min_tot_benes_when_present
0,11,0,11


The providers in Part D but not in Part B (section 7) are the closest measurable proxy
for what Part B is hiding: prescribers who clear Part D's threshold but have no Part B
service line that clears 11 beneficiaries. It is a lower bound on the censoring, not a
measurement of it — but it is the only handle the published data gives us.

## 9. Two loose ends from section 7 and 4b

### 9a. Is the Part-B-only population just organizations?

Section 7 found 468,667 NPIs in Part B but not Part D. That was read as a hint about
Part B's censoring, but there is a duller explanation available: Part B includes
**organizations** (`Rndrng_Prvdr_Ent_Cd = 'O'`) — labs, imaging centers, group practices
— and organizations do not write prescriptions, so their absence from Part D is expected
and says nothing about suppression.

Splitting the figure by entity type separates the two explanations. Only the
individual-provider share is evidence about censoring.

In [26]:
q("""
    WITH b AS (
        SELECT Rndrng_NPI AS npi,
               ANY_VALUE(Rndrng_Prvdr_Ent_Cd)       AS entity,
               COUNT(DISTINCT Rndrng_Prvdr_Ent_Cd)  AS n_entity_codes
        FROM part_b GROUP BY 1
    ),
    d AS (SELECT DISTINCT Prscrbr_NPI AS npi FROM part_d)
    SELECT
        b.entity,
        COUNT(*)                                                          AS part_b_providers,
        COUNT(*) FILTER (d.npi IS NULL)                                   AS part_b_only,
        ROUND(100.0 * COUNT(*) FILTER (d.npi IS NULL) / COUNT(*), 1)      AS pct_not_in_part_d,
        MAX(b.n_entity_codes)                                             AS max_entity_codes_per_npi
    FROM b LEFT JOIN d ON b.npi = d.npi
    GROUP BY 1
    ORDER BY 1
""", "part_b-only NPIs split by entity type (I = individual, O = organization)")

[   2.5s] part_b-only NPIs split by entity type (I = individual, O = organization)


,entity,part_b_providers,part_b_only,pct_not_in_part_d,max_entity_codes_per_npi
0,I,1113417,406803,36.5,1
1,O,61864,61864,100.0,1


### 9b. Brand or generic as the Part D peer key?

Section 4b found cost-per-claim varying up to 33× *within* a single generic name
(Fluticasone Propionate: $16.86 p50 → $565.79 p99), which suggests `Gnrc_Name` bundles
strengths, formulations and brands that are not really comparable on price. Section 6
showed brand is nearly free as a key: 0.71% → 0.91% of rows unscoreable.

If grouping by brand genuinely removes product-mix noise, the within-group cost spread
should tighten measurably. If it doesn't, the variation is prescriber behaviour after
all — which is the more interesting answer, and the one that keeps the generic key.

Measured on groups of ≥ 30, so the comparison is over peer groups we would actually
score.

In [27]:
q("""
    WITH gen AS (
        SELECT 'specialty + generic' AS key_def,
               COUNT(*) AS n,
               quantile_cont((Tot_Drug_Cst / NULLIF(Tot_Clms, 0))::DOUBLE, 0.50) AS p50,
               quantile_cont((Tot_Drug_Cst / NULLIF(Tot_Clms, 0))::DOUBLE, 0.99) AS p99
        FROM part_d GROUP BY Prscrbr_Type, Gnrc_Name HAVING COUNT(*) >= 30
    ),
    brn AS (
        SELECT 'specialty + brand' AS key_def,
               COUNT(*) AS n,
               quantile_cont((Tot_Drug_Cst / NULLIF(Tot_Clms, 0))::DOUBLE, 0.50) AS p50,
               quantile_cont((Tot_Drug_Cst / NULLIF(Tot_Clms, 0))::DOUBLE, 0.99) AS p99
        FROM part_d GROUP BY Prscrbr_Type, Brnd_Name HAVING COUNT(*) >= 30
    )
    SELECT
        key_def,
        COUNT(*)                                                   AS n_groups,
        SUM(n)                                                     AS n_rows_covered,
        ROUND(quantile_cont(p99 / NULLIF(p50, 0), 0.50), 2)        AS median_cost_p99_over_p50,
        ROUND(quantile_cont(p99 / NULLIF(p50, 0), 0.90), 2)        AS p90_cost_p99_over_p50,
        ROUND(MAX(p99 / NULLIF(p50, 0)), 2)                        AS max_cost_p99_over_p50
    FROM (SELECT * FROM gen UNION ALL SELECT * FROM brn)
    WHERE p50 IS NOT NULL AND p50 > 0
    GROUP BY key_def
    ORDER BY key_def
""", "within-group cost-per-claim spread: generic key vs brand key")

[   2.9s] within-group cost-per-claim spread: generic key vs brand key


,key_def,n_groups,n_rows_covered,median_cost_p99_over_p50,p90_cost_p99_over_p50,max_cost_p99_over_p50
0,specialty + brand,15277,26551252.0,2.87,5.86,172.50
1,specialty + generic,13085,26604270.0,3.16,9.74,409.43


One structural check to interpret the above: how many distinct brands does a generic
name actually cover? If most generics map to a single brand, the two keys are nearly the
same thing and any difference above is noise.

In [28]:
q("""
    WITH g AS (
        SELECT Gnrc_Name, COUNT(DISTINCT Brnd_Name) AS n_brands
        FROM part_d GROUP BY 1
    )
    SELECT
        COUNT(*)                                  AS generics,
        COUNT(*) FILTER (n_brands = 1)            AS with_one_brand,
        ROUND(AVG(n_brands), 2)                   AS mean_brands,
        quantile_cont(n_brands, 0.50)             AS p50_brands,
        quantile_cont(n_brands, 0.90)             AS p90_brands,
        MAX(n_brands)                             AS max_brands
    FROM g
""", "brands per generic name")

[   1.2s] brands per generic name


,generics,with_one_brand,mean_brands,p50_brands,p90_brands,max_brands
0,1779,1135,1.77,1.0,3.0,50


## Findings

All numbers below were measured by the cells above (project rule 7). Timings are
**warm-cache** — the files had been read earlier in the session — so they are a floor,
not a cold-start baseline. See "open" below.

### The peer key

Sections 3b and 6 together settle this. Adding a dimension to the peer key splits groups
that are already small, and the cost is measured as `pct_rows_under_30` — the share of
rows that fall into groups too small to score honestly:

| Peer key (Part B) | groups | median | % rows unscoreable at n<30 |
|---|---|---|---|
| specialty + code | 48,165 | 4 | **1.95%** |
| specialty + code + place of service | 59,087 | 3 | 2.40% |
| specialty + code + RUCA | 124,554 | 2 | 4.84% |
| specialty + code + state | 462,739 | 2 | **19.35%** |

Part D behaves the same way: generic alone costs 0.71%, brand 0.91%, and adding state
jumps to 8.75%.

- **State does not belong in the peer key.** It costs a tenth of the Part B dataset
  (1.95% → 19.35%) to buy a comparison that `Avg_Mdcr_Stdzd_Amt` already makes valid
  without it. State stays a *filter and display* dimension, not a grouping one.
- **RUCA is the better geographic refinement, if we want one.** It costs 4.84% rather
  than 19.35%, and it captures the mechanism that actually plausibly drives volume —
  urban vs. rural patient density — instead of an administrative boundary. This is the
  interesting middle option and it was not on the table before section 6.
- **Place of service is nearly free** (1.95% → 2.40%) and is part of Part B's natural
  grain, so it belongs in the key.
- **For Part D, brand beats generic as the peer key for cost.** Section 9b: switching
  from `Gnrc_Name` to `Brnd_Name` leaves coverage essentially unchanged (26,604,270 →
  26,551,252 rows in scoreable groups) but tightens the within-group cost-per-claim
  spread where it matters — median `p99/p50` 3.16 → 2.87, p90 **9.74 → 5.86**, max
  **409.43 → 172.50**. The median barely moves; the tail halves. So product-mix noise is
  real but concentrated in a minority of drugs, which is exactly what the structure
  predicts: 1,135 of 1,779 generics (63.8%) map to a single brand and are unaffected,
  while the rest average 1.77 brands and run to 50.
- Proposed default: **`(specialty, code, place_of_service)`** for Part B and
  **`(specialty, brand)`** for Part D, with `Gnrc_Name` kept as an attribute for rollup
  and browsing, and RUCA-refined peers as a documented secondary view.

### Sort / partition keys

- **Sort by `(specialty, code)` is confirmed.** Peer groups are tiny (median 4–5 rows)
  against a 122,880-row Parquet row group, and Part B's largest group (96,640) fits
  inside one. Predicted effect: a single-peer-group filter scans ~1 row group instead of
  ~79 (Part B) / ~218 (Part D). Every peer-key variant in section 6 has a smaller max
  group than the base key, so the bound holds for all of them.
- **Partition by year only.** Nothing in the cardinality argues for a second level.
- **Pin identifier types explicitly in the Parquet build.** `Rndrng_Prvdr_Zip5` and both
  `State_FIPS` columns happened to infer as `VARCHAR`, so leading zeros survived — luck,
  not a guarantee. `Rndrng_Prvdr_RUCA` inferred as `DOUBLE` but is a categorical code
  (22 values); float equality on it will eventually bite, and section 6 now makes it a
  candidate peer-key column, which raises the stakes.
- `Tot_Srvcs` is `DOUBLE`, not an integer, and its true minimum is **5.5** — Part B
  service counts really are fractional.

### One provider dimension, not two

Section 7 came back cleaner than expected:

- **706,614 NPIs appear in both datasets** — 60.1% of Part B's 1,175,281 providers and
  64.0% of Part D's 1,104,162 prescribers. 468,667 are Part B only, 397,548 Part D only,
  for a union of **1,572,829** distinct providers.
- **No provider carries more than one specialty within a dataset.** `max_specialties` is
  1 in both, for every NPI. So specialty is a well-defined provider-level attribute and
  belongs on a provider dimension, not repeated on every fact row.
- **The specialty strings agree exactly, 706,614 out of 706,614 — 100%.** The mismatch
  query returned an empty result.

  This corrects an earlier reading in this notebook. Part B having 104 specialty values
  and Part D 175 looked like two taxonomies needing a crosswalk. It is **one** taxonomy:
  Part D simply exercises more of it, because it covers prescriber types (dentists,
  optometrists) that have no Part B service line. **No crosswalk is needed**, and the
  schema gets a single `provider` dimension keyed on NPI with one specialty column.

### Suppression — both datasets censor at 11, on different measures

- **Part B: `MIN(Tot_Benes)` is exactly 11, with zero rows below.** The low end is
  477,670 rows at 11, 423,446 at 12, 380,812 at 13, decreasing monotonically. A natural
  distribution would keep *rising* toward the low end; this one peaks at the threshold
  and stops. The distribution has been cut, not tapered — hard confirmation of the
  documented rule.
- **Part D censors on claims, not beneficiaries**: `MIN(Tot_Clms)` is 11 with zero rows
  below, and where `Tot_Benes` survives it is also ≥ 11.
- So the two datasets drop rows on different criteria, and only Part D *also* blanks a
  column. Part D loses `Tot_Benes` on **55.08%** of rows (14,757,509 of 26,794,878) and
  `GE65_Tot_Benes` on 88%. `Tot_Clms` and `Tot_Drug_Cst` are never null — build Part D
  metrics on those.
- **Part B's censoring is invisible and that is the dangerous part.** It produces no
  nulls at all (zero in `Tot_Benes`, `Tot_Srvcs`, `Avg_Mdcr_Stdzd_Amt`, specialty,
  state). A provider who performs a procedure 8 times does not appear as missing data —
  they appear as a provider who does not perform it. Low-volume providers are
  systematically under-represented and nothing in the file says so. The dashboard must
  state this, because a user reading a provider profile is looking at a censored record.
- **Entity type explains only a small part of the Part-B-only population.** Section 9a:
  of the 468,667 NPIs in Part B but not Part D, **61,864 are organizations** — and
  *100.0%* of Part B's organizations are absent from Part D, which is exactly right,
  since labs and imaging centers do not prescribe. But that accounts for only 13.2% of
  the gap. The remaining **406,803 are individuals** (36.5% of Part B's 1,113,417
  individual providers).

  That residual is **not** clean evidence of censoring, because it is confounded with
  specialty: surgeons, radiologists, pathologists and physical therapists legitimately
  write few or no Part D prescriptions. The honest conclusion is that the Part-B-only
  count cannot be used to size Part B's censoring without first conditioning on
  specialty. Recorded as a dead end rather than a result.
- (Consistency check: 406,803 + 61,864 = 468,667, matching section 7. Every NPI carries
  exactly one entity code.)

### What "outlier" should mean

- **Mean and standard deviation are unusable on the raw measures.** `mean_over_median` is
  6.4 for Part B `Tot_Srvcs` and **12.4 / 12.7** for Part D `Tot_Drug_Cst` and
  cost-per-claim. The extremes are real: one Part B provider-code row carries $299M of
  total payment, one Part D row $81.8M of drug cost. A z-score would measure the tail
  against itself. **Use rank/percentile or median-and-MAD.**
- **Within a peer group, price is nearly fixed and volume is not.** Across the 25 largest
  Part B groups, `price_p99/p50` on `Avg_Mdcr_Stdzd_Amt` is **1.00–1.23** — Medicare sets
  the rate — while `srvcs_p99/p50` runs **4.1–14.2**. The Part B signal is *utilization*.
- **Where price does vary, that is itself the finding.** Diagnostic Radiology / 71046
  (chest X-ray) is the sole exception in the top 25 at **3.34**, against ~1.1 elsewhere.
- **Part D peer groups are not cost-homogeneous.** Cost-per-claim within one generic:
  Fluticasone Propionate $16.86 → $565.79 p99 (**33×**), Levothyroxine 7×, vs ~2× for
  Amoxicillin. `Gnrc_Name` bundles strengths, formulations and brands, so a Part D cost
  outlier may be product mix rather than prescriber behaviour — and section 9b confirms
  it partly is, halving the p90 within-group cost spread when grouped by brand instead.

### Other schema notes

- **"State" has 62 values**, not 50 — territories, military codes, foreign addresses. 395
  Part B rows are non-US. The geography reference table must handle these deliberately.
- 7,600 Part B rows have a null RUCA code — relevant now that RUCA is a peer-key
  candidate.

### CSV baseline timings — the "before" for lever 1

Single full-file scans: **~1.2–1.5 s** for both files. The seven-pass distribution
queries took **9.1 s** (Part B) and **10.7 s** (Part D), consistent at ~1.3 s per pass.
Section 6's multi-key profiles: 4.8 s (four Part B aggregations) and 3.9 s (three Part D).
Section 7's distinct-NPI set operations: 2.4 s each. Environment: DuckDB 1.5.5, 15
threads, 19.1 GiB memory limit.

**This materially changes the framing of lever 1.** The case-study plan assumed
CSV → Parquet would be "the largest single win." At ~1.3 s per scan of a 2.9 GB CSV,
DuckDB's parallel CSV reader is already near-interactive on a single pass, and the honest
headline is more likely sort order and the pre-aggregated peer-stat table. Report what
the measurement says.

### Open

- **Cold-cache timings.** Everything above ran warm. The Parquet comparison needs both
  formats measured the same way, cold and warm, or the speedup is meaningless.
- **Sizing Part B's censoring** would need the Part-B-only individuals broken down by
  specialty, to separate "suppressed" from "does not prescribe." Worth doing only if the
  writeup needs to quantify the bias rather than just disclose it.
